BƯỚC 1: TẢI FASTTEXT

In [1]:
# [BƯỚC 1] TẢI TÀI NGUYÊN (FASTTEXT)
import os

# Kiểm tra file đã tồn tại chưa
if not os.path.exists('crawl-300d-2M.vec'):
    print("⏳ Đang tải FastText (Chờ 1-2 phút)...")
    !wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip

    print("⏳ Đang giải nén...")
    !unzip -q crawl-300d-2M.vec.zip
    print("✅ Tải xong!")
else:
    print("✅ File FastText đã có sẵn!")

⏳ Đang tải FastText (Chờ 1-2 phút)...
⏳ Đang giải nén...
✅ Tải xong!


BƯỚC 2: TIỀN XỬ LÝ & TOKENIZATION

In [2]:
# [BƯỚC 2] TIỀN XỬ LÝ & SỐ HÓA (TOKENIZATION)
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# CẤU HÌNH CHO LSTM
MAX_WORDS = 20000       # Từ điển 20k từ
MAX_LEN = 250           # Độ dài câu 250
EMBEDDING_DIM = 300     # FastText 300d

print("⏳ Đang load và làm sạch dữ liệu...")

# Load Data
data_path = 'MBTI 500.csv'
if not os.path.exists(data_path):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if '500' in filename: data_path = os.path.join(dirname, filename); break

df = pd.read_csv(data_path)
df.dropna(subset=['posts', 'type'], inplace=True)

# Hàm làm sạch (Giữ nguyên logic cũ)
def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    mbti_types = ['infj', 'intp', 'infp', 'entp', 'intj', 'istj', 'entj', 'istp',
                  'enfj', 'enfp', 'isfj', 'esfj', 'isfp', 'estj', 'estp', 'esfp']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)
    # Giữ lại chữ, số và các dấu câu cơ bản để LSTM hiểu ngữ cảnh (nếu cần)
    # Nhưng để đơn giản và nhất quán, ta vẫn lọc chỉ lấy chữ/số
    return re.sub(r'[^a-z0-9\s]', '', text).strip()

df['clean_text'] = df['posts'].apply(basic_clean)

# Tokenization
print("⏳ Đang số hóa (Tokenizing)...")
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])
sequences = tokenizer.texts_to_sequences(df['clean_text'])
X_data = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

word_index = tokenizer.word_index
print(f"✅ Dữ liệu X đã xong: {X_data.shape}")

⏳ Đang load và làm sạch dữ liệu...
⏳ Đang số hóa (Tokenizing)...
✅ Dữ liệu X đã xong: (73036, 250)


BƯỚC 3: TẠO MA TRẬN EMBEDDING (FASTTEXT)


In [3]:
# [BƯỚC 3] TẠO MA TRẬN TRỌNG SỐ TỪ FASTTEXT
print("⏳ Đang trích xuất Vector FastText (Quá trình này mất khoảng 1 phút)...")

embeddings_index = {}
# Đọc file từng dòng để tiết kiệm RAM
with open('crawl-300d-2M.vec', encoding='utf8') as f:
    for line in f:
        values = line.rstrip().rsplit(' ')
        word = values[0]
        if word in word_index:
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs

print(f"   -> Tìm thấy {len(embeddings_index)} vector từ trùng khớp.")

# Tạo ma trận khởi tạo
num_words = min(MAX_WORDS, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))

for word, i in word_index.items():
    if i >= MAX_WORDS: continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

print(f"✅ Ma trận Embedding sẵn sàng: {embedding_matrix.shape}")

⏳ Đang trích xuất Vector FastText (Quá trình này mất khoảng 1 phút)...
   -> Tìm thấy 103745 vector từ trùng khớp.
✅ Ma trận Embedding sẵn sàng: (20000, 300)


BƯỚC 4: XÂY DỰNG Bi-LSTM + ATTENTION VÀ TRAIN

In [4]:
# [BƯỚC 4] HUẤN LUYỆN MODEL CHIẾN LƯỢC (Bi-LSTM + ATTENTION)
from tensorflow.keras.layers import Layer, Input, Embedding, Bidirectional, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

# --- 1. ĐỊNH NGHĨA LỚP ATTENTION (CUSTOM LAYER) ---
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        # Tạo trọng số W và b cho Attention
        self.W = self.add_weight(name='attention_weight',
                                 shape=(input_shape[-1], 1),
                                 initializer='normal',
                                 trainable=True)
        self.b = self.add_weight(name='attention_bias',
                                 shape=(input_shape[1], 1),
                                 initializer='zeros',
                                 trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        # Tính điểm số (score) cho từng từ
        e = K.tanh(K.dot(x, self.W) + self.b)
        # Chuyển thành xác suất (weights) bằng Softmax
        a = K.softmax(e, axis=1)
        # Nhân trọng số với vector từ đầu vào
        output = x * a
        # Cộng tổng lại để ra vector ngữ cảnh duy nhất (Context Vector)
        return K.sum(output, axis=1)

# --- 2. HÀM DỰNG MODEL ---
def build_bilstm_attention_model():
    inputs = Input(shape=(MAX_LEN,), dtype='int32')

    # Embedding: Dùng FastText, đóng băng (trainable=False)
    x = Embedding(input_dim=num_words, output_dim=EMBEDDING_DIM,
                  weights=[embedding_matrix], input_length=MAX_LEN, trainable=False)(inputs)

    # Spatial Dropout: Xóa nguyên 1 chiều vector (Tốt cho NLP)
    x = tf.keras.layers.SpatialDropout1D(0.3)(x)

    # Bi-LSTM Layer 1: return_sequences=True để truyền cho lớp sau
    x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0))(x)

    # Bi-LSTM Layer 2: Vẫn return_sequences=True để Attention tính toán từng từ
    x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0))(x)

    # Attention Mechanism: "Nhìn" vào toàn bộ chuỗi để chọn ý chính
    x = Attention()(x)

    # Dense Layers
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = BatchNormalization()(x) # Giúp hội tụ ổn định hơn
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# --- 3. TRAINING LOOP ---
axes = {'IE': ('I', 'E'), 'NS': ('N', 'S'), 'TF': ('F', 'T'), 'JP': ('J', 'P')}
lstm_results = []

print("🚀 BẮT ĐẦU HUẤN LUYỆN Bi-LSTM + ATTENTION...")
print("=" * 80)

for axis, (char0, char1) in axes.items():
    print(f"\n🔥 TRỤC: {axis} ({char0} vs {char1})")

    # Chuẩn bị dữ liệu
    y_data = df['type'].apply(lambda x: 1 if char1 in x else 0).values
    X_train, X_val, y_train, y_val = train_test_split(
        X_data, y_data, test_size=0.1, random_state=42, stratify=y_data
    )

    # Tính Class Weights (Cân bằng dữ liệu)
    weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weights = {0: weights[0], 1: weights[1]}

    # Reset RAM
    tf.keras.backend.clear_session()
    model = build_bilstm_attention_model()

    # Early Stopping (Kiên nhẫn 4 epoch vì LSTM học chậm nhưng chắc)
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

    # Train
    print("   -> Đang train (Model này hơi lâu, ông kiên nhẫn nhé)...")
    model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=15,
              batch_size=128, # Batch lớn để chạy nhanh hơn trên GPU
              class_weight=class_weights,
              callbacks=[early_stop],
              verbose=1)

    # Đánh giá
    y_prob = model.predict(X_val)
    y_pred = (y_prob > 0.5).astype(int)

    acc = accuracy_score(y_val, y_pred)
    f1_target = f1_score(y_val, y_pred, pos_label=1)
    mf1 = f1_score(y_val, y_pred, average='macro')
    wf1 = f1_score(y_val, y_pred, average='weighted')

    lstm_results.append({'Trục': axis, 'Accuracy': acc, 'F1-Target': f1_target, 'Macro F1': mf1, 'Weighted F1': wf1})
    print(f"   ✅ Xong {axis}! Acc: {acc:.4f} | MF1: {mf1:.4f}")

# --- 4. HIỂN THỊ KẾT QUẢ ---
print("\n" + "="*80)
print("🏆 KẾT QUẢ CUỐI CÙNG: Bi-LSTM + ATTENTION (MBTI 500)")
print("="*80)
final_df = pd.DataFrame(lstm_results)
final_df = final_df[['Trục', 'Accuracy', 'F1-Target', 'Macro F1', 'Weighted F1']]
pd.options.display.float_format = '{:.4f}'.format
print(final_df)

🚀 BẮT ĐẦU HUẤN LUYỆN Bi-LSTM + ATTENTION...

🔥 TRỤC: IE (I vs E)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Đang train (Model này hơi lâu, ông kiên nhẫn nhé)...
Epoch 1/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.5723 - loss: 0.7108 - val_accuracy: 0.7579 - val_loss: 0.5487
Epoch 2/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 54s 105ms/step - accuracy: 0.6807 - loss: 0.6111 - val_accuracy: 0.7281 - val_loss: 0.5656
Epoch 3/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7065 - loss: 0.5877 - val_accuracy: 0.7299 - val_loss: 0.5542
Epoch 4/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 49s 96ms/step - accuracy: 0.7103 - loss: 0.5766 - val_accuracy: 0.7953 - val_loss: 0.4860
Epoch 5/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.7345 - loss: 0.5623 - val_accuracy: 0.6841 - val_loss: 0.6258
Epoch 6/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7375 - loss: 0.5445 - val_accuracy: 0.7588 - val_loss: 0.5215
Epoch 7/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7427 - loss: 0.5354 - val_accuracy: 0.6821 - val_loss: 0.6199
Epoch 8/15
514/514 ━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Đang train (Model này hơi lâu, ông kiên nhẫn nhé)...
Epoch 1/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 55s 96ms/step - accuracy: 0.5315 - loss: 0.7274 - val_accuracy: 0.8087 - val_loss: 0.4704
Epoch 2/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 81s 94ms/step - accuracy: 0.6448 - loss: 0.6184 - val_accuracy: 0.7000 - val_loss: 0.5985
Epoch 3/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7081 - loss: 0.5976 - val_accuracy: 0.8146 - val_loss: 0.4548
Epoch 4/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.6976 - loss: 0.5902 - val_accuracy: 0.3624 - val_loss: 0.9329
Epoch 5/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7258 - loss: 0.5647 - val_accuracy: 0.6609 - val_loss: 0.6188
Epoch 6/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7054 - loss: 0.5574 - val_accuracy: 0.6413 - val_loss: 0.6302
Epoch 7/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7310 - loss: 0.5405 - val_accuracy: 0.7681 - val_loss: 0.4951
229/229 ━━━━━━━━━━━━━━━━━━━━ 5s

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Đang train (Model này hơi lâu, ông kiên nhẫn nhé)...
Epoch 1/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 53s 95ms/step - accuracy: 0.6204 - loss: 0.6339 - val_accuracy: 0.8439 - val_loss: 0.3958
Epoch 2/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.7588 - loss: 0.5165 - val_accuracy: 0.6758 - val_loss: 0.6555
Epoch 3/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.7812 - loss: 0.4810 - val_accuracy: 0.8608 - val_loss: 0.3337
Epoch 4/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 49s 95ms/step - accuracy: 0.7911 - loss: 0.4620 - val_accuracy: 0.8497 - val_loss: 0.3705
Epoch 5/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.8019 - loss: 0.4532 - val_accuracy: 0.8011 - val_loss: 0.4242
Epoch 6/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.8040 - loss: 0.4352 - val_accuracy: 0.7685 - val_loss: 0.5133
Epoch 7/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.8171 - loss: 0.4269 - val_accuracy: 0.8536 - val_loss: 0.3441
229/229 ━━━━━━━━━━━━━━━━━━━━ 5s

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Đang train (Model này hơi lâu, ông kiên nhẫn nhé)...
Epoch 1/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 55s 96ms/step - accuracy: 0.5302 - loss: 0.7281 - val_accuracy: 0.4578 - val_loss: 0.7544
Epoch 2/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.5988 - loss: 0.6622 - val_accuracy: 0.6431 - val_loss: 0.6316
Epoch 3/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.6243 - loss: 0.6495 - val_accuracy: 0.6653 - val_loss: 0.6115
Epoch 4/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.6340 - loss: 0.6393 - val_accuracy: 0.6804 - val_loss: 0.5985
Epoch 5/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.6456 - loss: 0.6314 - val_accuracy: 0.6914 - val_loss: 0.5871
Epoch 6/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.6614 - loss: 0.6180 - val_accuracy: 0.6416 - val_loss: 0.6314
Epoch 7/15
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.6692 - loss: 0.6103 - val_accuracy: 0.6978 - val_loss: 0.5776
Epoch 8/15
514/514 ━━━━━━━━━━━━